# MOB analysis
For the maintained download-and-run example without Seurat or CARD, use `inst/examples/MOB.R` as documented in the README. This notebook is the extended workflow for generating markers with Seurat and plotting with CARD, which require separate installations.

In [ ]:
#load the necessary packages
library(data.table)
library(QRSIDE)
library(SingleCellExperiment)
library(Seurat)

#load the datasets
load("../scdata/MOB/MOB.dge.sceset.RData")
load("../spdata/MOB/Rep12_MOB_count_matrix-1.RData")
load("../spdata/MOB/Rep12_MOB_manual_annotation.RData")

count_matrix=as.data.frame(t(assays(sce)$counts))
meta = as.data.frame(colData(sce))
meta['cellType'][meta['cellType']=='M/TC']='M_TC'
rownames(count_matrix)=paste0('cell', 1:nrow(count_matrix))
rownames(meta)=paste0('cell', 1:nrow(meta))

#construct the SeuratObject for scdata
seu <- CreateSeuratObject(counts = t(count_matrix))
active_ident=as.matrix(meta[,'cellType'])
colnames(active_ident)=NULL
rownames(active_ident)= rownames(meta)
Idents(seu)=active_ident[,1]
cell_names=rownames(as.matrix(Idents(seu)))

In [ ]:
#Quality control of the spatial transcriptomic data and corresponding spatial spots
celltype=c('GC','PGC','M_TC','OSNs','EPL-IN')
Num_celltype=length(celltype)

geneexpr=as.matrix(t(MOB_raw))
rownames(geneexpr) = paste0("spot",1:nrow(geneexpr))

pos <- cbind.data.frame(x=as.numeric(sapply(strsplit(colnames(MOB_raw),split="x"),"[",1)),y=as.numeric(sapply(strsplit(colnames(MOB_raw),split="x"),"[",2)))
colnames(pos) = c("x", "y")
rownames(pos) = paste0("spot",1:nrow(pos))


QR_res=QRSIDE_QC(geneexpr,pos,20)

In [ ]:
#Run DEG analysis on scdata to identify top 50 marker genes for each cell types

selected_sc_count = as.matrix(SeuratObject::GetAssayData(seu, assay = "RNA", layer = "counts"))
selected_sc_meta =as.data.frame(Idents(seu))

colnames(selected_sc_meta) = "cellType"
selected_sc_meta$sampleInfo = rownames(selected_sc_meta)

selection_seu <- CreateSeuratObject(counts = selected_sc_count,min.features = 200,min.cells=3)

active_ident=as.matrix(selected_sc_meta[,'cellType'])
colnames(active_ident)=NULL
rownames(active_ident)= rownames(selected_sc_meta)

Idents(selection_seu)=active_ident[,1]
cell_names=rownames(as.matrix(Idents(selection_seu)))

selection_seu = SCTransform(selection_seu)
all.markers <- FindAllMarkers(selection_seu)

library(dplyr)
all.markers %>%
  group_by(cluster) %>%
  top_n(n = 50, wt = avg_log2FC) -> top50

top_DEGs =dplyr::rename(top50,'label'='cluster')
top_DEGs=top_DEGs %>% arrange(label, desc(avg_log2FC))  

In [ ]:
#cell-type deconvolution using QR-SIDE
library(QRSIDE)

sp_expr=QR_res[[1]] # Spot by gene SRT matrix

sp_pos=QR_res[[2]] #Spot by 2, 2D spatial coordination

top_DEGs=top_DEGs #DEGs dataframe from Seurat sorted by 'avg_log2FC'

Num_topic=7 #Num of topic domains

Num_HVG=1000 #num of highly variable genes used in QR_SIDE

dim_embed=20 #dimension of embeddings in hierachical factor models

top_marker_num=4 #Num of markers of each cell type from "top_DEGs" used in QR_SIDE 

fixed_marker_list=FALSE #whether QR-SIDE uses the all genes in "top_DEGs" as markers. If FALSE, "top_marker_num" must be provided

out=run_QRSIDE(sp_expr, sp_pos, top_DEGs, Num_topic, Num_HVG, dim_embed, top_marker_num, fixed_marker_list)

In [ ]:
#visualzie the cell-type proportion

clusters=c()
pi = out$pi
beta = out$beta
proportion = matrix(0,nrow(pi),ncol(beta))

for (i in 1:nrow(pi)){
 idx = which.max(pi[i,])
 clusters=append(clusters,idx) #spot-separable cluster label
 proportion[i,] = beta[idx,] #spot-separable cell type proportion
}

colnames(proportion)=celltype[out$order]

color=c('#E0E0EE','#DF4D4F','#F08B21','#D0B14C','#4374A4')

rownames(proportion) = rownames(sp_pos)

library(CARD)
CARD.visualize.pie(proportion = proportion,spatial_location = sp_pos, color=color, radius=0.45)